# DrugForge — Benchmarks (Colab runner)

Runs the heavy redocking benchmarks on Colab CPU so the 8 GB VPS is never
loaded, then produces two JSON artifacts to upload to the server:

- `internal.json` — PfDHFR redocking RMSD, reproducibility, controls, enrichment
- `external.json` — independent redocking over the fixed Astex complex list

The server only serves these artifacts read-only at `/api/benchmarks`.

**Honest scope.** These are in-silico benchmarks of a docking pipeline, not a
clinical or experimental validation. Skipped complexes are reported, not hidden.

## 1. Install dependencies

In [ ]:
!pip -q install rdkit meeko vina scipy gemmi requests
!apt-get -qq install -y openbabel > /dev/null
print('dependencies installed')

## 2. Get the code

Either clone the repository, or upload the `drugforge/`, `scripts/` and
`data/benchmark_sets/` directories. Set `REPO_URL` if you have a remote.

In [ ]:
import os

REPO_URL = ''  # e.g. 'https://github.com/you/drugforge.git'

if REPO_URL:
    !git clone -q $REPO_URL drugforge_repo
    os.chdir('drugforge_repo')
else:
    from google.colab import files
    print('Upload a zip of the project (drugforge/, scripts/, data/benchmark_sets/)')
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.zip'):
            !unzip -q -o $name -d drugforge_repo
            os.chdir('drugforge_repo')
            break

print('cwd:', os.getcwd())
!ls

## 3. Internal validation (PfDHFR)

Roughly 5-15 minutes: redocking, 5 repeats, 2 controls, 36 enrichment compounds.

In [ ]:
!python scripts/bench_internal.py --exhaustiveness 8 --repeats 5

## 4. External redocking

Start with `--limit 5` to confirm the pipeline works, then run the full 85
complexes. Expect roughly 1-3 hours on a Colab CPU runtime.

In [ ]:
!python scripts/bench_external.py --limit 5 --exhaustiveness 8 --output /tmp/external_smoke.json

In [ ]:
!python scripts/bench_external.py --exhaustiveness 8

## 5. Inspect the artifacts before publishing

In [ ]:
import json, pathlib

for name in ('internal', 'external'):
    path = pathlib.Path(f'data/benchmarks/{name}.json')
    if not path.exists():
        print(f'{name}: not generated')
        continue
    data = json.loads(path.read_text())
    print(f'--- {name} ---')
    if name == 'internal':
        print('redocking     ', data['redocking'].get('results'))
        print('reproducibility', data['reproducibility'].get('spread'), 'kcal/mol spread')
        print('ordering_held ', data['controls'].get('ordering_held'))
        e = data['enrichment']
        if e.get('status') == 'available':
            print('vina AUC      ', e['vina']['auc'])
            print('consensus AUC ', e['consensus']['auc'])
            print('improves      ', e['consensus_improves'])
    else:
        print('attempted     ', data['attempted'])
        print('evaluated     ', data['evaluated'])
        print('skipped       ', data['skipped'], data['skip_reasons'])
        print('rmsd<2A       ', data['success_rate_under_2A'])
        print('median rmsd   ', data['median_rmsd'])

## 6. Download the artifacts, then upload them to the server

In [ ]:
from google.colab import files
import pathlib

for name in ('internal', 'external'):
    path = pathlib.Path(f'data/benchmarks/{name}.json')
    if path.exists():
        files.download(str(path))

Copy them into the deployment, then reload the page:

```bash
scp internal.json external.json root@SERVER:/opt/drugforge/data/benchmarks/
```

No server restart is required: `/api/benchmarks` reads the files on each request.